In [0]:
df_rotas = spark.read.parquet("/Volumes/workspace/dundermiffin/logistica/rotas_otimizadas.parquet")
df_rotas.createOrReplaceTempView("rotas_otimizadas")

df_entrega = spark.read.parquet("/Volumes/workspace/dundermiffin/logistica/entrega.parquet")
df_entrega.createOrReplaceTempView("entrega")

df_armazem = spark.read.parquet("/Volumes/workspace/dundermiffin/source/armazens.parquet")
df_armazem.createOrReplaceTempView("armazem")

df_clientes = spark.read.parquet("/Volumes/workspace/dundermiffin/source/clientes.parquet")
df_clientes.createOrReplaceTempView("clientes")

##Análise de compreensão - Rotas

### Célula 3
- Adiciona uma flag pra sinalizar rotas com duração impossível (usando 10h como jornada de trabalho de referência)
### Célula 4
- Baseado nas informações da célula 3 o armazém 4 (Norte) parece ser o mais problemático. Norte possui uma malha rodoviaria complicada: Cidades longes umas das outras e poucas rodovias diretas o que aumenta o tempo de entrega tornando-o inviável
- Se a média de qtd_entregas das rotas inviáveis for bem maior que a das viáveis, confirma que na verdade o problema é volume de clientes por rota, não distância isolada.
### Célula 5
- Confirma as informaçõe da célula 4, mostra que São Paulo, não Belém, é o armazém com mais jornadas inviáveis. Demonstrando que o problema não é a distância e sim a má distribuição de entregas entre rotas
### Célula 6
- Mostra que São Paulo não possui um número gigante de entregas então provavelmente ele está fazendo, dentro de uma mesma rota, viagens muito longas.
### Célula 7
- Tentando compreender se o problema está na regra de cobertura regional. Onde Sudeste atende Sudeste e Sul. A query mostra quantas UFs são atendidas por armazém
- Confirma que o problema não é a quantidade de estados atendidos, já que São Paulo nem é o armazém que atende mais UFs
### Célula 8
- Mostra dados muito mais problemáticos: Rotas com 1 única entrega já têm 30% de inviabilidade com uma média de 14 horas de viagem
### Célula 9
- Tenta compreender as distancias percorridas por cada armazém. E quantos clientes com distâncias longas cada um atende
- Confirma a informação sobre o Norte. Um único cliente está a mais de 4 mil Kms do armazém, impossibilitando que a entrega seja feita em uma única jornada
- Para São Paulo mostra que mais da metade dos clientes estão a uma distância que já é fisicamente inviável de percorrer (ida) em menos de 10h dirigindo.

###Estas análises já são comprovação que a regra de regiões permitidas tem furos. 
Próxima análise pretende compreender se a melhor escolha é apenas mudar a regra, realocar um dos armazéns ou abrir mais um armazém mais proximo ao centro oeste


In [0]:
%sql
SELECT
    id_armazem,
    data_saida,
    ROUND(SUM(duracao_percorrida_min) / 60, 2) AS horas_totais_rota,
    COUNT(*) AS qtd_entregas,
    CASE
        WHEN SUM(duracao_percorrida_min) / 60 > 10 THEN 'ROTA INVIÁVEL'
        ELSE 'OK'
    END AS status_jornada
FROM rotas_otimizadas
GROUP BY id_armazem, data_saida
ORDER BY horas_totais_rota DESC


In [0]:
%sql
SELECT
    status_jornada,
    COUNT(*) AS qtd_rotas,
    ROUND(AVG(qtd_entregas), 1) AS media_entregas,
    ROUND(AVG(horas_totais_rota), 1) AS media_horas
FROM (
    SELECT
        id_armazem,
        data_saida,
        ROUND(SUM(duracao_percorrida_min) / 60, 2) AS horas_totais_rota,
        COUNT(*) AS qtd_entregas,
        CASE
            WHEN SUM(duracao_percorrida_min) / 60 > 10 THEN 'ROTA INVIÁVEL'
            ELSE 'OK'
        END AS status_jornada
    FROM rotas_otimizadas
    GROUP BY id_armazem, data_saida
)
GROUP BY status_jornada

In [0]:
%sql
SELECT
    id_armazem,
    COUNT(*) AS total_rotas,
    SUM(CASE WHEN horas_totais_rota > 10 THEN 1 ELSE 0 END) AS rotas_inviaveis,
    ROUND(100.0 * SUM(CASE WHEN horas_totais_rota > 10 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_inviavel
FROM (
    SELECT
        id_armazem,
        data_saida,
        SUM(duracao_percorrida_min) / 60 AS horas_totais_rota
    FROM rotas_otimizadas
    GROUP BY id_armazem, data_saida
)
GROUP BY id_armazem
ORDER BY pct_inviavel DESC

In [0]:
%sql
SELECT
    id_armazem,
    COUNT(DISTINCT id_cliente) AS total_clientes_unicos,
    COUNT(*) AS total_entregas,
    ROUND(AVG(qtd_entregas_por_rota), 1) AS media_entregas_por_rota
FROM (
    SELECT
        id_armazem,
        id_cliente,
        data_saida,
        COUNT(*) OVER (PARTITION BY id_armazem, data_saida) AS qtd_entregas_por_rota
    FROM rotas_otimizadas
)
GROUP BY id_armazem
ORDER BY total_clientes_unicos DESC

In [0]:
%sql
SELECT
    e.id_armazem,
    COUNT(DISTINCT c.regiao) AS qtd_regioes_atendidas,
    COUNT(DISTINCT c.uf) AS qtd_ufs_atendidas
FROM entrega e
JOIN clientes c ON c.id = e.id_cliente
GROUP BY e.id_armazem
ORDER BY qtd_ufs_atendidas DESC

In [0]:
%sql
SELECT
    qtd_entregas,
    COUNT(*) AS qtd_rotas,
    ROUND(AVG(horas_totais_rota), 1) AS media_horas,
    ROUND(100.0 * SUM(CASE WHEN horas_totais_rota > 10 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_inviavel
FROM (
    SELECT
        id_armazem,
        data_saida,
        COUNT(*) AS qtd_entregas,
        SUM(duracao_percorrida_min) / 60 AS horas_totais_rota
    FROM rotas_otimizadas
    GROUP BY id_armazem, data_saida
)
GROUP BY qtd_entregas
ORDER BY qtd_entregas

In [0]:
%sql
SELECT
    id_armazem,
    COUNT(*) AS total_clientes,
    ROUND(AVG(distancia_km), 0) AS media_distancia_km,
    ROUND(MAX(distancia_km), 0) AS maior_distancia_km,
    SUM(CASE WHEN distancia_km > 800 THEN 1 ELSE 0 END) AS clientes_acima_800km
FROM entrega
GROUP BY id_armazem
ORDER BY media_distancia_km DESC